## 一、引言

随着人工智能技术的变革，单一模态的模型已经难以满足复杂场景下的交互需求，视觉-语言跨模态模型(LLM+Vision)成为业界研究和关注的核心方向。这类模型旨在融合视觉信息和语言信息，实现如图像描述、视觉问答、多模态对话等通用且复杂的任务。

BLIP-2、LLaVA与MiniGPT-4无疑是视觉-语言跨模态领域的代表性模型。尽管三者在模型架构的细节设计上各有侧重，但均采用了"视觉特征对齐-语言生成能力/多模态指令微调"的两阶段训练策略，为探索视觉与语言模型的深度融合提供了极具参考价值的技术范式。深入剖析这三大模型的技术路径可以发现，它们既面临着跨模态领域的共性核心问题，也在各自的优化方向上呈现出独特的技术思考。

它们共同面临的首要核心问题，是**模态鸿沟的跨越与特征表示的高效融合**。预训练视觉编码器(如ViT、CLIP)与大语言模型(如Flan-T5、Vicuna)是在完全不同的数据分布和任务目标下训练而成的：视觉编码器专注于提取图像的空间特征与视觉语义，输出的是高维密集特征向量；而大语言模型则擅长捕捉文本的序列依赖与语义关联，其输入为离散的token序列。如何设计高效的"桥梁模块"(如BLIP-2的Q-Former、LLaVA的线性投影层)来对齐和融合这两种截然不同的特征表示，是实现视觉-语言联合理解与生成的关键。

第二个共同挑战在于**轻量化高效训练策略的设计与成本控制**。由于大语言模型通常包含数十亿参数，直接联合训练会带来巨大的计算和存储开销。BLIP-2、LLaVA与MiniGPT-4均采用冻结预训练模型参数，仅训练连接模块和微调少量大语言模型的策略(LLaVA)，大幅降低了训练成本。同时，它们还通过精心设计的预训练任务(如图像-文本对比、匹配与生成)和微调数据集(如指令跟随数据)来提升模型在复杂多模态任务中的表现。

相较于BLIP-2更侧重于模型架构层面的特征融合创新，LLaVA与MiniGPT-4在实践中进一步关注到了**高质量微调数据对模型能力**(尤其是指令跟随能力/语言连贯性)的增益作用，这也成为两者区别于其他早期模型的重要技术特征。例如，LLaVA利用GPT-4生成了15万条视觉-语言指令对的数据集；MiniGPT-4则利用一阶段模型和GPT-4辅助生成了高质量多模态指令数据，进一步提升了模型在复杂视觉问答和多轮对话中的表现。这些实践充分证明，高质量、多样化的微调数据能够有效提升模型对复杂指令的理解能力、视觉细节的捕捉能力以及回答的准确性与逻辑性，是弥补模型"模态偏见"、提升交互自然度的重要支撑。

## 二、核心技术对比

|对比维度|BLIP-2|LLaVA|MiniGPT-4|
|---|---|---|---|
|**视觉编码器**|冻结预训练ViT系列，支持ViT-L/14、ViT-g/14等|LLaVA 1.0用CLIP ViT-L/14-224px；<br>LLaVA 1.5升级为CLIP ViT-L/14-336px|使用冻结的BLIP-2预训练组件(EVA-CLIP中的ViT-g/14 和 Q-Former)|
|**大语言模型（LLM）**|冻结开源LLM，如OPT（解码器）、Flan-T5（编解码器）|Vicuna7B(解码器)|冻结的Vicuna等(解码器)|
|**跨模态连接模块**|轻量级Querying Transformer（Q-Former），含32个768维查询向量, 学习视觉-语言表征，再用线性层将查询向量映射成语言模型相同维度且作为软提示加入到LLM中 | LLaVA 1.0用单一线性层；<br>LLaVA 1.5升级为两层线性层+激活函数的MLP结构| 冻结的Q-Former + 线性投影层 |
|**核心训练数据集**|预训练：COCO, Visual Genome, CC3M, CC12M,SBU, 115M from LAION-400M；（使用CapFilt方法创建人工注解）|预训练：CC3M筛选的595K图像-文本对；<br>微调：158K指令跟随数据|预训练：5M弱标注图像-文本对, 来自Conceptual Caption, SBU和LAION数据集；<br>微调：3.5K高质量多模态指令数据；Localized Narratives数据集|
|**训练阶段划分**|阶段一: 基于冻结视觉编码器的视觉-语言表示学习；<br>阶段二: 基于冻结LLM的视觉-语言生成学习|阶段一: 特征对齐预训练（训投影层）；<br>阶段二: 指令跟随数据集微调（微调训投影层+LLM）|阶段一: 视觉-语言特征对齐预训练（训投影层）；<br>阶段二: 指令微调（微调投影层）|
|**核心损失函数**|阶段1：<br>图像-文本对比损失（ITC）双向交叉熵、<br>图像-文本匹配损失（ITM）二元交叉熵、<br>图像-文本生成损失（ITG）交叉熵；<br>阶段2：文本生成交叉熵损失|预训练：caption文本生成交叉熵损失；<br>微调：多轮对话只关注`<STOP>`和生成回答的部分内容计算损失|预训练：文本生成损失；<br>微调：指令跟随生成损失（不包含指令部分）|
|**训练成本**| 一阶段250k步，二阶段80k步，16卡A100(40G) 训练6天（一阶段）+ 3天（二阶段）|8卡A100完成全流程训练（预训练+微调）约18小时，1M量级数据即可支撑|预训练: 20,000训练步，4卡A100(80G) 10小时 <br>微调: 400训练步，1卡A100 7分钟|


<!-- 
|**训练成本**| （ViT-L 474M， ViT-g 1.2B）一阶段250k步，二阶段80k步，batchsize(2320/1680 for ViT-L/ViT-g)(1920/1520 for OPT/Flan-T5) 16-A100(40G) 训练少于6天（一阶段）+3天（二阶段）|8卡A100完成全流程训练（预训练+微调）约18小时，1M量级数据即可支撑|预训练: 20,000训练步，batchsize=256, 10小时(4张A100(80G)) <br>微调: 400训练步， batchsize=12, 7分钟(单 A100)|
 -->

### 网络架构

- BLIP-2 网络结构：使用冻结Image Encoder和可以训练的Q-Former来提取图像（和文本最匹配）的视觉特征表示$Z$，然后通过线性投影层将$Z$映射到大语言模型（LLM）所需的输入维度，最后将映射后的特征作为软提示加入到LLM中进行多模态理解与生成任务。
<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;text-align:center;">
<image src="./assets/blip-2_arch.png" />
<span style="font-size:12px; color:#555;">图1. BLIP-2 网络结构</span>
</div> 

- LLaVA 网络结构：直接用一个线性投影层$W$将视觉编码器的输出映射到语言模型的词嵌入空间。

<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;text-align:center;">
<image src="./assets/llava_arch1.png" />
<span style="font-size:12px; color:#555;">图2. LLaVA 网络结构</span>
</div> 

- MiniGPT-4 网络结构：采用冻结的BLIP-2预训练组件(EVA-CLIP中的ViT-g/14 和 Q-Former)作为视觉编码器，通过线性投影层将Q-Former输出的视觉特征映射到LLM的输入空间。

<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;text-align:center;">
<image src="./assets/minigpt4_arch.png" />
<span style="font-size:12px; color:#555;">图3. MiniGPT-4 网络结构</span>
</div> 

### 训练策略

__BLIP-2:__

- 阶段一：视觉-语言特征对齐预训练（冻结视觉编码器，仅训练Q-Former）
  - 数据集：和BLIP相同（COCO, Visual Genome, CC3M, CC12M,SBU, 115M from LAION-400M）；使用CapFilt方法来创建人工注解。
  - 损失函数：使用大规模图像-文本对，通过聚合图像-文本对比损失(ITC)、图像-文本匹配损失(ITM)和图像-文本生成损失(ITG)三种损失来优化模型
  - 目标：使视觉特征与语言特征在语义空间中对齐，输出和文本最相关的视觉特征表示。
<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;text-align:center;">
<image src="./assets/blip-2_stage1.png" />
<span style="font-size:12px; color:#555;">图4. BLIP-2 一阶段结构</span>
</div> 

- 阶段二：视觉-语言生成预训练（冻结视觉编码器和LLM，仅训练Q-Former和线性投影层）
  - 损失函数：使用图像-文本对，通过文本生成交叉熵损失来优化模型
  - 目标：使其能够生成与图像内容相关的自然语言描述。
<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;text-align:center;">
<image src="./assets/blip-2_stage2.png" />
<span style="font-size:12px; color:#555;">图5. BLIP-2 二阶段结构</span>
</div> 

__LLaVA:__

- 阶段一: 特征对齐的预训练 (更新$W$)
  - 数据集：使用CC3M筛选的595K图像-文本对
  - 损失函数：caption文本生成交叉熵损失
  - 目标：将视觉特征与语言模型的词嵌入空间对齐，使得视觉信息能够被语言模型有效利用
- 阶段二: 端到端微调 (更新$W$和语言模型参数)
  - 数据集：利用GPT-4生成的158K视觉-语言指令跟随数据
  - 损失函数：多轮对话只关注`<STOP>`和生成回答的部分内容计算交叉熵损失
  - 目标：提升模型在复杂多模态任务中的理解和生成能力

__MiniGPT-4:__

- 阶段一: 特征对齐预训练 (更新线性投影层)
  - 数据集：Conceptual Caption, SBU和LAION数据集中的5M弱标注图像-文本对
  - 损失函数：文本生成损失
  - 出现的问题：生成不连贯的语言输出，如重复的词语、句子或无关内容（从GPT3.5受到启发，决定使用指令微调来解决这些问题）
- 阶段二: 指令微调 (更新线性投影层)
  - 数据集：使用一阶段模型生成的3.5K高质量多模态指令数据(通过GPT-4辅助)；Localized Narratives数据集
  - 损失函数：指令跟随生成损失（不包含指令部分）
  - 目标：优化一阶段出现的问题，使模型产生更自然、可靠的语言输出

## 三、结论

__创新点__

1. BLIP-2：BLIP-2的核心突破在于其引入**Q-Former**这一轻量的跨模态桥梁模块。在冻结预训练视觉编码器和大语言模型的前提下，BLIP-2通过训练Q-Former实现了高效的视觉-语言特征对齐与融合，显著降低了训练成本。二是其提出了二阶段训练范式，先是视觉-语言表示预训练，然后是融合图像信息的语言生成能力的训练，为后续跨模态模型提供了重要参考。

2. LLaVA：LLaVA的创新聚焦于“轻量链接模块”和“高质量的指令微调数据集”相结合。在模型层面，它采用简单高效的线性投影层(LLaVA1.5使用两个线性层的MLP)作为视觉-语言特征对齐的桥梁，结构更简洁且易于与开源大语言模型(如Vicuna)适配，降低了技术落地门槛；在数据层面，LLaVA通过GPT-4构建了“视觉指令调优数据集(LLaVA-Instruct-150K)”，通过“图像-注解-指令”的三元组数据形式，将通用图像描述转化为符合人类交互习惯的指令任务，使模型能够精准理解并回复复杂指令，显著提升了多模态对话的自然度与准确性。同时在微调过程中，LLaVA同时训练了连接模块和部分LLM参数，进一步增强了模型的适应能力。

3. MiniGPT-4：MiniGPT-4和LLaVA类似。两者都注重第二阶段的指令微调并都利用GPT-4辅助生成了高质量的多模态指令数据。但MiniGPT-4使用了更多的数据在一阶段的预训练中完成视觉-语言特征对齐，例如一阶段结束后，模型已经可以用来生成图像的详细描述并使用GPT-4辅助修改、纠正错误，从而为二阶段微调提供更高质量的指令数据集。此外，MiniGPT-4也尝试了不同的连接模块，如线性投影层/不带Q-Former的版本等等。和LLaVA本质的不同是没有微调LLM本身，节约了计算资源。

__各模型的主要局限性__

1. BLIP-2：BLIP-2的性能高度依赖所对接的大语言模型能力，若搭配的LLM（如Flan-T5）指令理解能力较弱，其多模态对话的灵活性会显著下降；由于使用的是CapFilt生成的注解，数据集本身质量可能不高。同时，由于其训练数据以通用图像描述为主，缺乏针对性的指令调优数据，在处理“图像编辑建议”“场景逻辑分析”等复杂指令时，响应的精准度与逻辑性不足。此外，Q-Former模块的训练需要依赖大规模视觉-文本预训练数据，对数据质量与数量的要求较高，增加了技术复现难度。当然，也继承了LLM本身的可能存在冒犯性语言、社会偏见、隐私泄漏等问题。

2. LLaVA：LLaVA采用的单个线性投影层虽结构简洁，但融合能力相对有限，难以充分捕捉图像中的细粒度特征，在处理“包含多个相似物体的场景识别”“微小视觉细节描述”等任务时表现较弱；LLaVA1.5中通过尝试更复杂的MLP结构，并使用更高分辨率的视觉编码器以及13B的Vicuna模型，得到了一定提升。但还存在一个根本问题: bag of patches， 即视觉编码器输出的patch特征是无序的，线性层难以捕捉空间关系，导致对复杂场景的理解能力受限。此外，LLaVA在微调过程中同时更新连接模块和部分LLM参数，虽然提升了模型适应性，但也增加了训练复杂度和计算资源需求。

3. MiniGPT-4：MiniGPT-4的大规模数据筛选与一阶段特征对齐训练时，需要消耗大量的计算资源，尤其是在使用高分辨率视觉编码器时，训练周期与成本显著高于LLaVA；此外，MiniGPT-4的视觉感知有限，难以区分空间定位。(可以通过RefCOCO和Visual Genome数据集进行训练)；以及MiniGPT-4继承了LLM本身的幻觉问题，且通过实验发现更长的回复往往伴随更高的幻觉率。（潜在优化方向：使用AI幻觉检测模块进行强化学习）